# Group 12 — PLSA Information Retrieval Project (All 20 Categories)
This notebook trains PLSA on **all 20 categories** of 20 Newsgroups. It implements PLSA with NumPy/EM, inspects topics, tests controlled mixtures from two selected categories, performs query folding-in, and compares retrieval against TF-IDF/VSM and LSI.

**Main experiment:** all 20 categories. **Boundary experiment:** a controlled mixture of two source categories (baseball and politics), as specified in the proposal. Category labels are not used to train PLSA; they are used for evaluation and selecting synthetic source documents.


All shared implementations are imported from `src/`. Run this notebook from the repository root. Old execution outputs were cleared during refactoring. The final cell writes tables and metadata for `Result.py`; experiment-specific loops remain here.

## 0. Install dependencies

Run once in a notebook environment if required. In Colab, use the first line in a code cell.

In [ ]:
# Install from the repository root: pip install -r requirements.txt
# Then: python -m spacy download en_core_web_sm


## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

from src.config import EXPERIMENTS, SUPERGROUPS, CATEGORY_QUERIES, SUPERGROUP_QUERIES, QUERY_BANK
from src.dataset import load_corpus
from src.preprocessing import preprocess_texts, split_train_test, split_train_val_test, vectorize_counts
from src.plsa_model import PLSA, train_plsa_restarts
from src.retrieval import fit_baselines, plsa_query_log_likelihood, background_word_probabilities
from src.metrics import metrics_at_k
from src.experiments import make_mixed_document, average_rank_score
from src.results import export_results
EXPERIMENT = 'PLSA_Baseline'
CONFIG = EXPERIMENTS[EXPERIMENT]


## 2. Load the full 20-category corpus

Labels define splits and relevance only; PLSA trains on counts without labels.

In [ ]:
# Shared loader; corpus selection remains an explicit experimental choice.
corpus = load_corpus(CONFIG["corpus"], seed=SEED)
documents, labels, target_names = corpus.documents, corpus.labels, corpus.target_names
original_labels = corpus.original_labels
source_category_names = corpus.source_category_names
source_to_supergroup = {category: group for group, members in SUPERGROUPS.items() for category in members}
print("Corpus:", CONFIG["corpus"], "Documents:", len(documents))
for i, label in enumerate(target_names):
    print(label, int((labels == i).sum()))


> Historical interpretation from before the shared-code refactor; rerun to verify any numerical claims below.

### Main corpus versus boundary experiment

The model and retrieval evaluation use the **full 20-category corpus**. The controlled boundary experiment still uses two chosen source categories to create interpretable 90:10 through 10:90 word mixtures. This does not mean the PLSA model was trained on only those two categories.

## 3. Train/test split and preprocessing

Split the raw documents first. Fit the vocabulary only on the training split to avoid vocabulary leakage from the test documents.

In [ ]:
# Split raw text first. Fit vocabulary only on training text.
docs_train, docs_test, y_train, y_test = split_train_test(documents, labels, test_size=CONFIG["test_size"], seed=SEED)
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer, X_train, X_test = vectorize_counts(docs_train_processed, docs_test_processed)
vocabulary = count_vectorizer.get_feature_names_out()
nonempty_test_mask = np.asarray([bool(doc.strip()) for doc in docs_test_processed])
print("Train/test count matrices:", X_train.shape, X_test.shape)
print("Non-empty test texts:", int(nonempty_test_mask.sum()))


## 4. PLSA theory

For document \(d\), word \(w\), and latent topic \(z\):

\[
P(w\mid d)=\sum_{z=1}^{K}P(w\mid z)P(z\mid d)
\]

- \(\theta_{d,z}=P(z\mid d)\): document-topic distribution.
- \(\phi_{z,w}=P(w\mid z)\): topic-word distribution.
- Each row of \(\theta\) and \(\phi\) sums to one.

For every observed document-word pair, the E-step computes the posterior responsibility:

\[
P(z\mid d,w)=\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'}P(z'\mid d)P(w\mid z')}
\]

The M-step uses the expected word counts to update both distributions. The process repeats until the log-likelihood stops improving.

In [ ]:
# The single EM and fold-in implementation lives in src/plsa_model.py.
from src.plsa_model import PLSA


## 5. Multiple random restarts

EM can settle at different local optima. Fit several initializations and select the model with the largest training log-likelihood. This selection criterion is only about training fit; final retrieval quality must still be evaluated on held-out documents.

In [ ]:
# train_plsa_restarts is imported from src above.

K = 5
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)

## 6. Inspect topics and convergence

Topic IDs are arbitrary: topic 0 is not inherently sports or politics. Interpret each topic by its highest-probability words, and use category-level averages only as a diagnostic.

In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Controlled topic-boundary experiment

Generate artificial documents by sampling word tokens from source documents in known categories. We then infer topic mixtures with folding-in and compare the known source proportions with the estimated probabilities.

**Interpretation caution:** a source-word ratio is not guaranteed to equal a latent-topic probability. Topics can split a category or overlap categories. We therefore identify sports-associated and politics-associated topics from training-set category averages, and interpret the result as an experiment rather than ground truth equivalence.

In [ ]:
sports_class = target_names.index("rec.sport.baseball")
politics_class = target_names.index("talk.politics.misc")

class_topic_means = np.vstack([
    theta_train[y_train == c].mean(axis=0)
    for c in range(len(target_names))
])
sports_topic = int(class_topic_means[sports_class].argmax())
politics_topic = int(class_topic_means[politics_class].argmax())

print("Sports-associated topic:", sports_topic)
print("Politics-associated topic:", politics_topic)
if sports_topic == politics_topic:
    print("Warning: both categories map to the same dominant topic; interpret estimates cautiously.")


# make_mixed_document is imported from src above.

boundary_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
boundary_rows = []

for ratio in ratios:
    for replicate in range(20):
        mixed_X = make_mixed_document(
            X_train, y_train, sports_class, politics_class,
            ratio, boundary_rng, n_tokens=300
        )
        estimate = best_plsa.fold_in(mixed_X, max_iter=100)[0]
        boundary_rows.append({
            "source_category_a": "rec.sport.baseball",
            "source_category_b": "talk.politics.misc",
            "actual_sports_word_ratio": ratio,
            "actual_politics_word_ratio": 1-ratio,
            "estimated_sports_topic": estimate[sports_topic],
            "estimated_politics_topic": estimate[politics_topic],
            "replicate": replicate
        })

boundary_df = pd.DataFrame(boundary_rows)
boundary_summary = boundary_df.groupby("actual_sports_word_ratio").agg(
    estimated_sports_mean=("estimated_sports_topic", "mean"),
    estimated_sports_std=("estimated_sports_topic", "std"),
    estimated_politics_mean=("estimated_politics_topic", "mean"),
    estimated_politics_std=("estimated_politics_topic", "std")
).reset_index()
display(boundary_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], "k--", label="Ideal reference")
plt.errorbar(
    boundary_summary["actual_sports_word_ratio"],
    boundary_summary["estimated_sports_mean"],
    yerr=boundary_summary["estimated_sports_std"],
    marker="o", capsize=3, label="Estimated sports-topic probability"
)
plt.xlabel("Actual sports word-token proportion")
plt.ylabel("Estimated sports-associated topic probability")
plt.title("PLSA topic-boundary experiment")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. TF-IDF/VSM and LSI baselines

All baselines are fitted on training documents only. LSI uses truncated SVD on TF-IDF representations; it is a geometric low-rank representation, unlike PLSA's normalized probability distributions.

In [ ]:
# All baselines use the shared train-only fitting implementation.
tfidf_vectorizer, lsi_model, tfidf_splits, lsi_splits = fit_baselines(
    docs_train_processed, docs_test_processed, seed=SEED
)
X_train_tfidf, X_test_tfidf = tfidf_splits
lsi_train, lsi_test = lsi_splits


## 9. Retrieval metrics and fair comparison

For each query, relevance is defined using the known test-category labels. The same test collection and relevance class are used for PLSA, LSI and TF-IDF. This is a small demonstration with category queries; a stronger report should evaluate a broader, predeclared query set and report macro averages.

**NDCG@10 replaces Recall@10 and F1@10** in this evaluation. Each document has binary relevance (1 if its label matches the query category; 0 otherwise). NDCG rewards relevant documents near the top and normalizes against an ideal ranking. With binary relevance, it does not distinguish degrees of relevance within the same category.


Evaluation reports **Precision@10, Recall@10, NDCG@10, and MAP@10**. MAP@10 is the average of AP@10 over all category queries.

In [ ]:
category_queries = {name: CATEGORY_QUERIES[name] for name in target_names}

missing = sorted(set(target_names) - set(category_queries))
extra = sorted(set(category_queries) - set(target_names))
print("Categories without a query:", missing)
print("Query names not found in dataset:", extra)

def evaluate_query(query_text, target_label, k=10):


    processed_query = preprocess_texts([query_text])[0]
    if not processed_query.strip():
        print(f"Warning: query for {target_names[target_label]} became empty after preprocessing.")


    q_tfidf = tfidf_vectorizer.transform([processed_query])
    q_lsi = lsi_model.transform(q_tfidf)


    q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    q_plsa = best_plsa.fold_in(q_counts, max_iter=100)


    sim_tfidf = cosine_similarity(q_tfidf, X_test_tfidf).ravel()
    sim_lsi = cosine_similarity(q_lsi, lsi_test).ravel()
    sim_plsa = cosine_similarity(q_plsa, theta_test).ravel()

    methods = {
        "TF-IDF/VSM": sim_tfidf,
        "LSI": sim_lsi,
        "PLSA": sim_plsa
    }

    rows = []
    total_relevant = int((y_test == target_label).sum())
    if total_relevant == 0:
        return rows

    for method_name, similarities in methods.items():

        metric = metrics_at_k(similarities, y_test, target_label, k=k, suffix="k")
        rows.append({
            "query": query_text,
            "target_category": target_names[target_label],
            "method": method_name,
            "Precision@k": metric["Precision@k"],
            "NDCG@k": metric["NDCG@k"],
            "Recall@k": metric["Recall@k"],
            "AP@k": metric["AP@k"]
        })
    return rows

eval_rows = []
for category_name in target_names:
    eval_rows.extend(
        evaluate_query(
            category_queries[category_name],
            target_names.index(category_name),
            k=10
        )
    )

evaluation_df = pd.DataFrame(eval_rows)
display(evaluation_df)
print("\nMean metrics across all 20 category queries:")
display(evaluation_df.groupby("method")[["Precision@k", "Recall@k", "NDCG@k", "AP@k"]].mean().rename(columns={"AP@k": "MAP@k"}))

In [ ]:
mean_metrics = evaluation_df.groupby("method")[["Precision@k", "Recall@k", "NDCG@k", "AP@k"]].mean().rename(columns={"AP@k": "MAP@k"})
mean_metrics.plot(kind="bar", figsize=(9, 5), ylim=(0, 1), rot=0)
plt.title("Held-out retrieval comparison")
plt.ylabel("Score")
plt.xlabel("Method")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Model-order study: how does K affect the model?

The proposal asks us to study how the number of latent topics affects convergence and topic separation. Compare a few values of K. For a fair held-out comparison, infer test-document topic distributions with `fold_in`; don't use training-document representations to score test documents.

In [ ]:
k_rows = []
for k_topics in [2, 3, 5, 8, 10]:
    model = PLSA(
        n_topics=k_topics, max_iter=80, tol=1e-4,
        random_state=SEED, beta=1.0
    ).fit(X_train)
    k_rows.append({
        "K": k_topics,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "test_documents": X_test.shape[0]
    })

k_df = pd.DataFrame(k_rows)
display(k_df)

## 11. Optional extension: tempered E-step (exploratory)

Hofmann's paper introduces Tempered EM as an entropy-regularized fitting procedure and recommends evaluating held-out performance while changing the inverse temperature. The `beta` option in the class is a simplified tempered E-step, **not a full reproduction of the paper's TEM schedule**. Treat this section as exploratory only, and don't claim that it fully implements TEM.

For a proper TEM experiment, start with standard EM at beta=1, lower beta in small steps, run tempered iterations, and monitor held-out predictive likelihood/perplexity; stop when held-out performance deteriorates. A rigorous implementation should define and document the exact schedule and stopping rule.

In [ ]:
tempered_rows = []
for beta in [1.0, 0.9, 0.8, 0.7]:
    model = PLSA(
        n_topics=K, max_iter=80, tol=1e-4,
        random_state=SEED, beta=beta
    ).fit(X_train)
    tempered_rows.append({
        "beta": beta,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "warning": "Simplified tempered E-step; not full TEM"
    })

tempered_df = pd.DataFrame(tempered_rows)
display(tempered_df)

## 12. Limitations to state in the report

1. PLSA is trained on all 20 categories, but learned topics do not necessarily correspond one-to-one to the 20 human labels.
2. Synthetic source-word proportions in the baseball/politics boundary experiment are not identical to latent-topic probabilities.
3. Selecting the best training likelihood does not guarantee the best retrieval performance.
4. Category labels are only a proxy for relevance; manually judged query-document relevance would be stronger.
5. The 20 representative queries are manually authored, and query wording can affect ranking. Include the query set in the report.
6. Hofmann's paper used MED, CRAN, CACM and CISI collections; this project uses 20 Newsgroups and does not reproduce the paper's exact numerical results.
7. The optional beta experiment is a simplified tempered E-step unless the full TEM schedule and held-out stopping criterion are implemented.
8. A capped vocabulary and sparse count matrix are used to keep computation manageable.

## Export this run

Save result tables, convergence, topic words, settings and source hashes. `Result.py` reads these files without repeating model training.

In [ ]:
# This is reached only after all experiment cells complete successfully.
result_directory = export_results(EXPERIMENT, globals())
print("Results saved to:", result_directory)
